[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/095.%20The%20Supply%20Chain%20Network%20Design%20under%20Uncertainty/P95-Tier-9_executed.ipynb)

---


# 95. Supply Chain Network Design under Uncertainty
## Tier 9: Quantum Optimization and a QUBO Formulation

This notebook follows a QUBO-based, hybrid quantum-classical workflow: define a network-design instance, encode binary assignment decisions and penalties, sample candidate solutions, refine them with classical local search, and inspect the results. The implementation is an executable educational model, not a quantum-computing benchmark.

### Goals and workflow

We will formulate a quadratic unconstrained binary optimization (QUBO) objective, use a factorized classical proxy to illustrate the sampling and annealing steps, combine that sampling with classical one-bit local search, and assess feasibility and objective values. The proxy makes the workflow runnable for the 450 assignment variables in this example without allocating a quantum state vector of size $2^{450}$.

### Model assumptions and scope

- A binary variable indicates whether a facility serves a customer. Each customer requires one assignment unit; products are represented by the average transportation cost across the listed products.
- The fixed facility cost is allocated evenly across all customers as an assignment-level cost. This is a modeling proxy: it is not the exact fixed-charge cost of opening a facility.
- Facility capacities are measured in assignment units. Since each customer contributes one unit and every listed capacity exceeds the total number of customers, capacity is nonbinding in this instance.
- Customer assignment is encouraged, rather than guaranteed, by a QUBO penalty. A finite penalty can trade off constraint satisfaction against cost; feasibility must therefore be checked after optimization.
- Scenario probabilities are retained in the instance data but are not used in the objective. This example does not model scenario-dependent demand, costs, recourse decisions, or risk measures.
- The factorized sampler is a classical stochastic proxy. Its gate-inspired updates do not reproduce quantum amplitudes, entanglement, quantum annealing, or quantum speedup. No quantum hardware is used, and the experiments below cannot establish quantum advantage.

### QUBO model

Let $F$ be the set of facilities and $C$ the set of customers. For each facility-customer pair, $x_{fc}\in\{0,1\}$ equals 1 when facility $f$ serves customer $c$. Let $c_{fc}$ be the assignment-level cost, consisting of the allocated fixed cost and the average transportation cost across products. The demand penalty encourages exactly one facility assignment per customer:

$$\min_{x\in\{0,1\}^{|F||C|}}\quad \sum_{f\in F}\sum_{c\in C} c_{fc}x_{fc} + P\sum_{c\in C}\left(\sum_{f\in F}x_{fc}-1\right)^2. $$

Here, $P$ is the demand penalty weight. For a binary vector, expanding the square gives a linear contribution $-P x_{fc}$ for each assignment variable and a quadratic contribution $2P x_{fc}x_{gc}$ for each distinct pair of facilities serving the same customer; the constant $P|C|$ does not affect which solution minimizes the objective. In code, these terms are assembled in `QuboFormulation.formulate_qubo`, and the energy is evaluated as a linear term plus the stored upper-triangular quadratic terms.

The exact-one condition is soft because it is represented by a finite penalty. Capacity does not require a penalty term here: under the stated unit-load assumption, at most 30 assignments can reach any facility, while every listed capacity is at least 120. The model has $|F||C|=450$ binary decision variables. A dense QUBO matrix stores $O(n^2)$ entries, while this formulation has quadratic interactions only between facilities competing to serve the same customer.

### Interpreting the results

A lower QUBO energy is better for the encoded objective, but it does not by itself prove that every customer is assigned exactly once. The notebook separately reports assignment feasibility and capacity checks. Comparisons are limited to the same encoded QUBO objective; they are not evidence that a quantum method outperforms a classical one.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Fixed seeds make the generated instance and stochastic search reproducible.
np.random.seed(42)

## 1. Create the network-design instance

The instance supplies facility costs and capacities, product-dependent transportation costs, and scenario probabilities. The current objective uses facility and transportation costs; the scenario data are present for context but are outside this model's scope. A fixed seed makes the randomly perturbed transportation costs reproducible when the notebook is run from the beginning.

In [ ]:
@dataclass
class QuantumOptimizationInstance:
    """Input data for the QUBO network-assignment example."""

    facilities: List[int]
    customers: List[int]
    products: List[int]
    scenarios: List[int]
    fixed_costs: List[float]
    capacities: List[float]
    transport_costs: Dict[Tuple[int, int, int], float]
    scenario_probs: List[float]
    quantum_params: Dict


def create_quantum_instance() -> QuantumOptimizationInstance:
    """Build the fixed example instance and its reproducibility settings."""
    facilities = list(range(15))
    customers = list(range(30))
    products = list(range(5))
    scenarios = list(range(10))

    fixed_costs = [
        180, 220, 160, 250, 200, 170, 210, 190,
        240, 150, 205, 215, 195, 225, 185,
    ]
    capacities = [
        150, 180, 130, 200, 160, 140, 170, 150,
        190, 120, 155, 165, 145, 175, 135,
    ]

    transport_costs = {}
    for facility in facilities:
        for customer in customers:
            for product in products:
                distance_factor = abs(facility - customer // 2) * 0.8
                product_factor = 1.0 + product * 0.15
                base_cost = 2.5 + distance_factor + product_factor
                transport_costs[(facility, customer, product)] = max(
                    1.8, base_cost + np.random.normal(0, 0.3)
                )

    scenario_probs = [0.12, 0.15, 0.18, 0.15, 0.10, 0.08, 0.07, 0.05, 0.03, 0.07]
    quantum_params = {
        "num_qubits": len(facilities) * len(customers),
        "circuit_depth": 50,
        "annealing_time": 100,
        "temperature_start": 1000.0,
        "temperature_end": 0.001,
        "quantum_gates": ["H", "CNOT", "RZ", "RX"],
        "mixing_strength": 0.5,
        "cost_strength": 1.0,
    }

    return QuantumOptimizationInstance(
        facilities=facilities,
        customers=customers,
        products=products,
        scenarios=scenarios,
        fixed_costs=fixed_costs,
        capacities=capacities,
        transport_costs=transport_costs,
        scenario_probs=scenario_probs,
        quantum_params=quantum_params,
    )


quantum_instance = create_quantum_instance()
print("Network-design instance")
print(f"Facilities: {len(quantum_instance.facilities)}")
print(f"Customers: {len(quantum_instance.customers)}")
print(f"Products: {len(quantum_instance.products)}")
print(f"Uncertainty scenarios in input data: {len(quantum_instance.scenarios)}")
print(f"Assignment variables: {quantum_instance.quantum_params['num_qubits']}")
print(f"Configured circuit depth: {quantum_instance.quantum_params['circuit_depth']}")
print(
    "Scenario probabilities sum to "
    f"{sum(quantum_instance.scenario_probs):.2f}"
)

## 2. Formulate the QUBO

A QUBO expresses the objective using binary variables and linear and pairwise terms. The variable map gives every facility-customer decision a unique position in the binary vector. For each customer, the squared demand penalty discourages both no assignment and multiple assignments. Transportation costs are averaged over products, and fixed cost is allocated evenly across customers as stated in the model assumptions.

In [ ]:
class QuboFormulation:
    """Build and evaluate the assignment QUBO using an upper-triangular matrix."""

    def __init__(self, instance: QuantumOptimizationInstance):
        self.instance = instance
        self.num_variables = len(instance.facilities) * len(instance.customers)
        self.Q_matrix = np.zeros((self.num_variables, self.num_variables))
        self.linear_terms = np.zeros(self.num_variables)
        self.var_mapping = {}
        self.index_mapping = []

        index = 0
        for facility in instance.facilities:
            for customer in instance.customers:
                self.var_mapping[(facility, customer)] = index
                self.index_mapping.append((facility, customer))
                index += 1

        self._quadratic_i = np.array([], dtype=int)
        self._quadratic_j = np.array([], dtype=int)
        self._quadratic_values = np.array([], dtype=float)

    def formulate_qubo(
        self, penalty_weights: Dict[str, float]
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Add assignment costs and exact-one demand penalties."""
        if penalty_weights.get("demand", 0) <= 0:
            raise ValueError("The demand penalty must be positive.")

        self.Q_matrix.fill(0.0)
        self.linear_terms.fill(0.0)
        self._add_objective_terms()
        self._add_customer_demand_constraints(penalty_weights["demand"])

        # Store only nonzero upper-triangular terms to evaluate energies efficiently.
        self._quadratic_i, self._quadratic_j = np.nonzero(np.triu(self.Q_matrix, k=1))
        self._quadratic_values = self.Q_matrix[
            self._quadratic_i, self._quadratic_j
        ]
        return self.Q_matrix, self.linear_terms

    def _add_objective_terms(self) -> None:
        """Add allocated fixed costs and product-averaged transport costs."""
        customer_count = len(self.instance.customers)
        product_count = len(self.instance.products)

        for facility in self.instance.facilities:
            allocated_fixed_cost = self.instance.fixed_costs[facility] / customer_count
            for customer in self.instance.customers:
                average_transport_cost = sum(
                    self.instance.transport_costs[(facility, customer, product)]
                    for product in self.instance.products
                ) / product_count
                index = self.var_mapping[(facility, customer)]
                self.linear_terms[index] += allocated_fixed_cost + average_transport_cost

    def _add_customer_demand_constraints(self, penalty_weight: float) -> None:
        """Expand penalty_weight * (sum_f x_fc - 1)^2 for each customer."""
        for customer in self.instance.customers:
            customer_indices = [
                self.var_mapping[(facility, customer)]
                for facility in self.instance.facilities
            ]

            for index in customer_indices:
                self.linear_terms[index] -= penalty_weight

            # Binary identity x^2 = x leaves pairwise terms only for distinct variables.
            for position, index_i in enumerate(customer_indices):
                for index_j in customer_indices[position + 1:]:
                    self.Q_matrix[index_i, index_j] += 2.0 * penalty_weight

    def calculate_qubo_energy(self, binary_solution: np.ndarray) -> float:
        """Evaluate the QUBO energy for one binary assignment vector."""
        solution = np.asarray(binary_solution, dtype=float)
        if solution.shape != (self.num_variables,):
            raise ValueError("Solution length does not match the QUBO variable count.")
        if not np.all((solution == 0) | (solution == 1)):
            raise ValueError("A QUBO solution must contain only zeros and ones.")

        linear_energy = float(np.dot(self.linear_terms, solution))
        quadratic_energy = float(
            np.sum(
                self._quadratic_values
                * solution[self._quadratic_i]
                * solution[self._quadratic_j]
            )
        )
        return linear_energy + quadratic_energy

    def calculate_qubo_energies(self, binary_solutions: np.ndarray) -> np.ndarray:
        """Evaluate a batch of binary vectors using the same QUBO terms."""
        solutions = np.asarray(binary_solutions, dtype=float)
        if solutions.ndim != 2 or solutions.shape[1] != self.num_variables:
            raise ValueError("Solution batch has an incompatible shape.")
        if not np.all((solutions == 0) | (solutions == 1)):
            raise ValueError("QUBO solutions must contain only zeros and ones.")

        energies = solutions @ self.linear_terms
        if self._quadratic_values.size:
            energies += np.sum(
                solutions[:, self._quadratic_i]
                * solutions[:, self._quadratic_j]
                * self._quadratic_values,
                axis=1,
            )
        return energies

    def decode_solution(self, binary_solution: np.ndarray) -> Dict:
        """Decode selected variables and retain only unambiguous assignments."""
        solution = np.asarray(binary_solution)
        if solution.shape != (self.num_variables,):
            raise ValueError("Solution length does not match the QUBO variable count.")

        selected_by_customer = {customer: [] for customer in self.instance.customers}
        open_facilities = set()
        for (facility, customer), index in self.var_mapping.items():
            if solution[index] == 1:
                selected_by_customer[customer].append(facility)
                open_facilities.add(facility)

        customer_assignments = {
            customer: facilities[0]
            for customer, facilities in selected_by_customer.items()
            if len(facilities) == 1
        }
        return {
            "open_facilities": sorted(open_facilities),
            "customer_assignments": customer_assignments,
            "selected_facilities_by_customer": selected_by_customer,
        }

## 3. Sampling proxy

A full state-vector representation for $n$ qubits stores $2^n$ complex amplitudes, so it is not a practical representation for this 450-variable example. The sampler below instead stores one probability per binary variable and draws independent bits. Its Hadamard-, CNOT-, and rotation-inspired updates are heuristic probability updates, not quantum gates. This explicit limitation lets us demonstrate the optimization workflow without presenting classical sampling as a physical quantum simulation.

In [ ]:
class QuantumSimulator:
    """Factorized classical sampling proxy for the optimization workflow."""

    def __init__(self, num_qubits: int, params: Dict):
        self.num_qubits = num_qubits
        self.params = params
        self.quantum_state = None
        self.measurement_history = []

    def initialize_quantum_state(self) -> np.ndarray:
        """Initialize independent binary variables with equal probabilities."""
        self.quantum_state = np.full(self.num_qubits, 0.5, dtype=float)
        return self.quantum_state

    def apply_hadamard_gate(self, qubit: int) -> None:
        """Use an equal-probability update inspired by Hadamard initialization."""
        if self.quantum_state is None:
            self.initialize_quantum_state()
        self.quantum_state[qubit] = 0.5

    def apply_cnot_gate(self, control_qubit: int, target_qubit: int) -> None:
        """Apply a classical marginal update inspired by a CNOT bit flip."""
        if self.quantum_state is None:
            self.initialize_quantum_state()
        control_probability = self.quantum_state[control_qubit]
        target_probability = self.quantum_state[target_qubit]
        self.quantum_state[target_qubit] = (
            (1.0 - control_probability) * target_probability
            + control_probability * (1.0 - target_probability)
        )

    def apply_rotation_gate(
        self, qubit: int, angle: float, gate_type: str = "RZ"
    ) -> None:
        """Perturb a sampling probability; this is not a physical gate operation."""
        if self.quantum_state is None:
            self.initialize_quantum_state()
        direction = 1.0 if gate_type == "RX" else -1.0
        self.quantum_state[qubit] = np.clip(
            self.quantum_state[qubit] + direction * 0.05 * np.sin(angle),
            0.01,
            0.99,
        )

    def build_quantum_circuit(self, depth: int) -> np.ndarray:
        """Apply bounded gate-inspired updates to the factorized sampler."""
        if depth < 0:
            raise ValueError("Circuit depth must be nonnegative.")
        if self.quantum_state is None:
            self.initialize_quantum_state()

        active_qubits = min(self.num_qubits, 10)
        for layer in range(depth):
            if layer == 0:
                for qubit in range(active_qubits):
                    self.apply_hadamard_gate(qubit)
            if layer % 2 == 1:
                for qubit in range(max(0, active_qubits - 1)):
                    self.apply_cnot_gate(qubit, qubit + 1)
            if layer % 3 == 2:
                for qubit in range(active_qubits):
                    self.apply_rotation_gate(
                        qubit, np.random.uniform(0, 2 * np.pi), "RZ"
                    )
        return self.quantum_state

    def measure_quantum_state(self, num_shots: int = 1000) -> List[np.ndarray]:
        """Draw binary samples from independent per-variable probabilities."""
        if num_shots <= 0:
            raise ValueError("The number of shots must be positive.")
        if self.quantum_state is None:
            self.initialize_quantum_state()

        samples = (
            np.random.random((num_shots, self.num_qubits))
            < self.quantum_state[np.newaxis, :]
        ).astype(int)
        self.measurement_history.extend(samples)
        return list(samples)

    def get_measurement_statistics(self) -> Dict:
        """Summarize observed bit strings without implying quantum statistics."""
        if not self.measurement_history:
            return {"error": "No samples are available."}

        measurements = np.asarray(self.measurement_history, dtype=int)
        unique_solutions, counts = np.unique(
            measurements, axis=0, return_counts=True
        )
        top_indices = np.argsort(counts)[-5:][::-1]
        return {
            "total_measurements": len(measurements),
            "unique_solutions": len(unique_solutions),
            "top_solutions": unique_solutions[top_indices],
            "top_counts": counts[top_indices],
            "measurement_probabilities": counts[top_indices] / len(measurements),
        }

## 4. Annealing and hybrid optimization

The annealing loop draws candidates at successive temperatures, retains the lowest-energy sample, and biases the sampling probabilities toward that sample. The hybrid method begins with classical one-bit local search, alternates sampling and classical refinement, and retains the best solution encountered. Here, “annealing” describes the workflow being illustrated; because the sampler is a classical factorized proxy, the procedure is not quantum annealing.

In [ ]:
class QuantumAnnealingOptimizer:
    """Annealing-style search using the factorized sampling proxy."""

    def __init__(self, qubo_formulation: QuboFormulation, quantum_simulator: QuantumSimulator):
        self.qubo = qubo_formulation
        self.simulator = quantum_simulator
        self.optimization_history = []

    def quantum_annealing(
        self,
        Q_matrix: np.ndarray,
        linear_terms: np.ndarray,
        temperature_schedule: List[float],
        num_shots: int = 1000,
    ) -> Dict:
        """Sample and score candidates at each temperature in the schedule."""
        if Q_matrix.shape != (self.qubo.num_variables, self.qubo.num_variables):
            raise ValueError("QUBO matrix has an incompatible shape.")
        if linear_terms.shape != (self.qubo.num_variables,):
            raise ValueError("Linear-term vector has an incompatible shape.")
        if not temperature_schedule or any(t <= 0 for t in temperature_schedule):
            raise ValueError("The temperature schedule must contain positive values.")

        best_solution = None
        best_energy = float("inf")
        energy_history = []

        for step, temperature in enumerate(temperature_schedule):
            progress = step / max(1, len(temperature_schedule) - 1)
            circuit_depth = int(
                self.simulator.params["circuit_depth"] * (1.0 - progress)
            )
            self.simulator.build_quantum_circuit(circuit_depth)

            for qubit in range(min(self.qubo.num_variables, 10)):
                self.simulator.apply_rotation_gate(
                    qubit, temperature * 0.01, "RX"
                )

            measurements = np.asarray(
                self.simulator.measure_quantum_state(num_shots), dtype=int
            )
            energies = self.qubo.calculate_qubo_energies(measurements)
            best_index = int(np.argmin(energies))
            step_best_solution = measurements[best_index].copy()
            step_best_energy = float(energies[best_index])
            energy_history.append(step_best_energy)

            if step_best_energy < best_energy:
                best_energy = step_best_energy
                best_solution = step_best_solution.copy()

            self._bias_quantum_state(step_best_solution, temperature)

        final_config = self.qubo.decode_solution(best_solution)
        result = {
            "best_solution": best_solution,
            "best_energy": best_energy,
            "final_config": final_config,
            "energy_history": energy_history,
            "total_measurements": len(temperature_schedule) * num_shots,
        }
        self.optimization_history.append(result)
        return result

    def _bias_quantum_state(
        self, good_solution: np.ndarray, temperature: float
    ) -> None:
        """Move sampling probabilities gradually toward a good observed sample."""
        if self.simulator.quantum_state is None:
            return
        learning_rate = min(0.25, 0.01 / max(temperature, 0.001))
        self.simulator.quantum_state *= 1.0 - learning_rate
        self.simulator.quantum_state += learning_rate * good_solution
        self.simulator.quantum_state = np.clip(
            self.simulator.quantum_state, 0.01, 0.99
        )

    def create_temperature_schedule(
        self, start_temp: float, end_temp: float, steps: int
    ) -> List[float]:
        """Create an exponentially decreasing temperature schedule."""
        if start_temp <= 0 or end_temp <= 0 or steps < 2:
            raise ValueError("Temperatures must be positive and steps must be at least two.")
        return [
            start_temp * (end_temp / start_temp) ** (step / (steps - 1))
            for step in range(steps)
        ]


class HybridQuantumClassicalOptimizer:
    """Combine proxy sampling with classical one-bit local search."""

    def __init__(self, qubo_formulation: QuboFormulation, quantum_simulator: QuantumSimulator):
        self.qubo = qubo_formulation
        self.simulator = quantum_simulator
        self.quantum_optimizer = QuantumAnnealingOptimizer(
            qubo_formulation, quantum_simulator
        )
        self.optimization_history = []

    def hybrid_optimization(
        self,
        max_iterations: int = 10,
        classical_iterations: int = 100,
        quantum_shots: int = 500,
    ) -> Dict:
        """Alternate proxy sampling and classical refinement of QUBO solutions."""
        if max_iterations <= 0 or classical_iterations < 0 or quantum_shots <= 0:
            raise ValueError("Iteration counts and shot count must be valid positive values.")

        current_solution = self._classical_local_search(classical_iterations)
        current_energy = self.qubo.calculate_qubo_energy(current_solution)
        best_solution = current_solution.copy()
        best_energy = current_energy
        optimization_log = []

        temperature_schedule = self.quantum_optimizer.create_temperature_schedule(
            100.0, 0.1, 20
        )

        for iteration in range(max_iterations):
            quantum_result = self.quantum_optimizer.quantum_annealing(
                self.qubo.Q_matrix,
                self.qubo.linear_terms,
                temperature_schedule,
                quantum_shots,
            )
            quantum_solution = quantum_result["best_solution"]
            quantum_energy = quantum_result["best_energy"]

            refined_solution = self._classical_local_search(
                classical_iterations, quantum_solution
            )
            refined_energy = self.qubo.calculate_qubo_energy(refined_solution)
            if refined_energy < best_energy:
                best_solution = refined_solution.copy()
                best_energy = refined_energy

            if quantum_energy < current_energy:
                current_solution = quantum_solution.copy()
                current_energy = quantum_energy
                improvement_type = "sampling"
            else:
                current_solution = refined_solution.copy()
                current_energy = refined_energy
                improvement_type = "classical"

            optimization_log.append(
                {
                    "iteration": iteration + 1,
                    "quantum_energy": quantum_energy,
                    "classical_energy": refined_energy,
                    "best_energy": best_energy,
                    "improvement_type": improvement_type,
                }
            )

        final_config = self.qubo.decode_solution(best_solution)
        result = {
            "best_solution": best_solution,
            "best_energy": best_energy,
            "final_config": final_config,
            "optimization_log": optimization_log,
            "total_iterations": max_iterations,
        }
        self.optimization_history.append(result)
        return result

    def _classical_local_search(
        self,
        max_iterations: int,
        initial_solution: Optional[np.ndarray] = None,
    ) -> np.ndarray:
        """Accept a one-bit neighbor only when it lowers the QUBO energy."""
        if initial_solution is None:
            current_solution = np.random.randint(
                0, 2, self.qubo.num_variables
            )
        else:
            current_solution = np.asarray(initial_solution, dtype=int).copy()
            if current_solution.shape != (self.qubo.num_variables,):
                raise ValueError("Initial solution has an incompatible shape.")

        current_energy = self.qubo.calculate_qubo_energy(current_solution)
        for _ in range(max_iterations):
            neighbor_solution = current_solution.copy()
            flip_index = np.random.randint(0, self.qubo.num_variables)
            neighbor_solution[flip_index] = 1 - neighbor_solution[flip_index]
            neighbor_energy = self.qubo.calculate_qubo_energy(neighbor_solution)
            if neighbor_energy < current_energy:
                current_solution = neighbor_solution
                current_energy = neighbor_energy
        return current_solution

## 5. Build the model and run the hybrid search

The penalty value below is an input to the QUBO, not a proof that the resulting binary vector is feasible. The separate feasibility checks count customers with exactly one selected facility and verify each facility's assignment load against its capacity. The model's unit-load assumption makes the capacity checks straightforward to interpret.

In [ ]:
qubo_formulation = QuboFormulation(quantum_instance)
penalty_weights = {"demand": 20.0}
Q_matrix, linear_terms = qubo_formulation.formulate_qubo(penalty_weights)

quantum_simulator = QuantumSimulator(
    qubo_formulation.num_variables,
    quantum_instance.quantum_params,
)
hybrid_optimizer = HybridQuantumClassicalOptimizer(
    qubo_formulation, quantum_simulator
)

optimization_result = hybrid_optimizer.hybrid_optimization(
    max_iterations=8,
    classical_iterations=50,
    quantum_shots=300,
)

best_solution = optimization_result["best_solution"]
best_energy = optimization_result["best_energy"]
final_config = optimization_result["final_config"]
optimization_log = optimization_result["optimization_log"]

selected_counts = {
    customer: len(facilities)
    for customer, facilities in final_config[
        "selected_facilities_by_customer"
    ].items()
}
exactly_one_count = sum(count == 1 for count in selected_counts.values())
capacity_loads = {
    facility: sum(
        facility in selected_facilities
        for selected_facilities in final_config[
            "selected_facilities_by_customer"
        ].values()
    )
    for facility in quantum_instance.facilities
}
capacity_feasible = all(
    capacity_loads[facility] <= quantum_instance.capacities[facility]
    for facility in quantum_instance.facilities
)

assert best_solution.shape == (qubo_formulation.num_variables,)
assert np.all((best_solution == 0) | (best_solution == 1))
assert np.isclose(
    best_energy, qubo_formulation.calculate_qubo_energy(best_solution)
)

print("QUBO and hybrid-search summary")
print(f"Binary assignment variables: {qubo_formulation.num_variables}")
print(f"QUBO matrix shape: {Q_matrix.shape}")
print(f"Nonzero quadratic terms: {len(qubo_formulation._quadratic_values)}")
print(f"Best encoded energy: {best_energy:.2f}")
print(f"Customers assigned to exactly one facility: {exactly_one_count}/{len(quantum_instance.customers)}")
print(f"Capacity checks pass: {capacity_feasible}")
print(f"Facilities with at least one selected assignment: {final_config['open_facilities']}")

## 6. Compare candidate-generation procedures

The greedy, random-search, and hybrid candidates are evaluated using the same QUBO energy. Greedy assignment selects the facility with the lowest allocated fixed-plus-average-transport cost for each customer; it does not solve the QUBO globally. Random search draws binary vectors without enforcing one assignment per customer. The comparison therefore reports both energy and exact-one assignment feasibility, rather than treating energy alone as a complete measure of solution quality.

In [ ]:
def compare_with_classical_methods() -> Dict:
    """Compare greedy and random candidates with the hybrid result."""
    instance = qubo_formulation.instance

    greedy_solution = np.zeros(qubo_formulation.num_variables, dtype=int)
    for customer in instance.customers:
        best_facility = min(
            instance.facilities,
            key=lambda facility: qubo_formulation.linear_terms[
                qubo_formulation.var_mapping[(facility, customer)]
            ],
        )
        greedy_solution[
            qubo_formulation.var_mapping[(best_facility, customer)]
        ] = 1

    random_solutions = np.random.randint(
        0, 2, size=(1000, qubo_formulation.num_variables)
    )
    random_energies = qubo_formulation.calculate_qubo_energies(random_solutions)
    best_random_index = int(np.argmin(random_energies))
    best_random_solution = random_solutions[best_random_index]

    candidates = {
        "Greedy": greedy_solution,
        "Random search": best_random_solution,
        "Hybrid search": best_solution,
    }
    results = {}
    for method, solution in candidates.items():
        decoded = qubo_formulation.decode_solution(solution)
        exact_assignments = sum(
            len(facilities) == 1
            for facilities in decoded["selected_facilities_by_customer"].values()
        )
        results[method] = {
            "energy": qubo_formulation.calculate_qubo_energy(solution),
            "exactly_one_customers": exact_assignments,
            "open_facilities": len(decoded["open_facilities"]),
        }

    return results


comparison_results = compare_with_classical_methods()
comparison_table = pd.DataFrame(comparison_results).T
comparison_table.index.name = "Candidate-generation procedure"
display(comparison_table)

## 7. Visualize optimization and assignments

The plots show the best energy reported in each hybrid iteration, the encoded energies of the candidate-generation procedures, and the selected facility-customer variables. The assignment heatmap distinguishes an unselected pair from a selected pair; it does not imply that every selected vector satisfies the exact-one condition. The feasibility panel reports exact-one customer assignments and capacity checks separately.

In [ ]:
def visualize_quantum_optimization_results() -> None:
    """Plot search history, candidate energies, and assignment feasibility."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    fig.suptitle("QUBO Network-Assignment Search Results", fontsize=15)

    history_iterations = [entry["iteration"] for entry in optimization_log]
    history_best = [entry["best_energy"] for entry in optimization_log]
    axes[0, 0].plot(
        history_iterations, history_best, marker="o", label="Best energy so far"
    )
    axes[0, 0].set_xlabel("Hybrid iteration")
    axes[0, 0].set_ylabel("QUBO energy")
    axes[0, 0].set_title("Hybrid search progress")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    method_names = list(comparison_results)
    method_energies = [
        comparison_results[name]["energy"] for name in method_names
    ]
    axes[0, 1].bar(method_names, method_energies, color=["#78909C", "#64B5F6", "#81C784"])
    axes[0, 1].set_ylabel("QUBO energy")
    axes[0, 1].set_title("Candidate energies (lower is better)")
    axes[0, 1].tick_params(axis="x", rotation=15)
    axes[0, 1].grid(True, alpha=0.3, axis="y")

    assignment_matrix = np.zeros(
        (len(quantum_instance.facilities), len(quantum_instance.customers))
    )
    for (facility, customer), index in qubo_formulation.var_mapping.items():
        assignment_matrix[facility, customer] = best_solution[index]
    image = axes[1, 0].imshow(
        assignment_matrix, cmap="viridis", aspect="auto", vmin=0, vmax=1
    )
    axes[1, 0].set_xlabel("Customer index")
    axes[1, 0].set_ylabel("Facility index")
    axes[1, 0].set_title("Selected assignment variables")
    fig.colorbar(image, ax=axes[1, 0], ticks=[0, 1], label="Assignment variable")

    feasibility_values = [
        exactly_one_count / len(quantum_instance.customers) * 100,
        100.0 if capacity_feasible else 0.0,
    ]
    feasibility_labels = ["Customers assigned\nexactly once", "Capacity checks\npass"]
    axes[1, 1].bar(
        feasibility_labels, feasibility_values, color=["#4DB6AC", "#FFB74D"]
    )
    axes[1, 1].set_ylabel("Percentage (%)")
    axes[1, 1].set_title("Feasibility checks")
    axes[1, 1].set_ylim(0, 105)
    axes[1, 1].grid(True, alpha=0.3, axis="y")
    for index, value in enumerate(feasibility_values):
        axes[1, 1].text(index, value + 1, f"{value:.1f}%", ha="center")

    plt.tight_layout()
    plt.show()


visualize_quantum_optimization_results()

## 8. Scalability: state-vector memory growth

A full state-vector simulator stores one complex amplitude for each of the $2^n$ basis states. With 16 bytes per complex128 amplitude, its memory requirement is $16\cdot 2^n$ bytes, or $n+4$ in base-2 logarithmic bytes. This theoretical calculation explains why the notebook uses a factorized proxy; it does not measure runtime or demonstrate quantum speedup. The proxy uses storage proportional to the number of variables, but that difference alone says nothing about solution quality or comparative computational advantage.

In [ ]:
qubit_counts = np.array([10, 50, 100, 200, 400, 800])
log2_memory_bytes = qubit_counts + 4

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(qubit_counts, log2_memory_bytes, marker="o", color="#5C6BC0")
ax.set_xlabel("Number of qubits, n")
ax.set_ylabel("log2 of state-vector memory (bytes)")
ax.set_title("Theoretical memory growth of a full state-vector representation")
ax.grid(True, alpha=0.3)
ax.annotate(
    "For complex128 amplitudes: log2(bytes) = n + 4",
    xy=(qubit_counts[2], log2_memory_bytes[2]),
    xytext=(qubit_counts[2] + 60, log2_memory_bytes[2] - 100),
    arrowprops={"arrowstyle": "->"},
)
plt.tight_layout()
plt.show()

## 9. Key takeaways and limitations

1. **QUBO encoding:** Binary assignment variables and a squared exact-one penalty produce linear and pairwise terms that can be evaluated as a QUBO energy.
2. **Penalty interpretation:** A finite penalty encourages the intended constraint but does not guarantee feasibility. The returned candidate must be checked against the original assignment requirements.
3. **Hybrid search flow:** The example combines candidate sampling, energy evaluation, and one-bit classical refinement. The objective and the candidate-generation procedure are separate parts of the method.
4. **Model scope:** Facility fixed cost is allocated across assignments, capacity is nonbinding under the unit-load assumption, and uncertainty scenarios are not included in the objective. These assumptions limit the business conclusions that can be drawn.
5. **Computational interpretation:** The sampler is a factorized classical proxy, not a quantum circuit simulator or quantum annealer. Its results cannot support claims of quantum advantage, exponential speedup, or superior solution quality relative to classical methods.

A fuller supply-chain model could incorporate scenario-dependent data, explicit facility-opening decisions, and operational constraints. Those extensions would change the model and are not implemented in this notebook. Any such formulation would need its own variable definitions, QUBO encoding, penalty analysis, and feasibility validation.